**Конвертация таблицы из docx в csv**

In [2]:
!pip install python-docx

In [3]:
# Загрузка библиотек
#from docx import Document
import pandas as pd
import numpy as np
import os
import docx
import re

In [4]:
path = 'E:/IrkutskProject/Data/'
years = [2016, 2017, 2018, 2019, 2020]
original_names = {
    2016: 'soc_znach2016.docx',
    2017: 'soc_znach2017.docx',
    2018: 'soc_znach2018.docx',
    2019: 'soc_znach2019.docx',
    2020: 'soc_znach2020.docx'
}
original_url = {}
for year in years:
    original_path = f'{path}{year}/Original/'
    original_url[year] = original_path + original_names[year]
    
    if not os.path.exists(original_url[year]):
        print(f"Файл {original_url[year]} не найден по указанному пути")
    else:
        print(f"Файл {original_url[year]} найден по указанному пути")

Файл E:/IrkutskProject/Data/2016/Original/soc_znach2016.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2017/Original/soc_znach2017.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2018/Original/soc_znach2018.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2019/Original/soc_znach2019.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2020/Original/soc_znach2020.docx найден по указанному пути


In [5]:
def extract_tuberculosis_table(file_path, target_block):
    doc = docx.Document(file_path)
    target_found = False
    
    # Перебираем все элементы в теле документа
    for i, element in enumerate(doc.element.body):
        
        # Шаг 1: Ищем текст заголовка
        # Проверяем только абзацы ('p'), чтобы не искать внутри других таблиц случайно
        if not target_found and element.tag.endswith('p'):
            text = "".join(element.itertext()).strip()
            # display(text)
            if target_block.lower() in text.lower():
                target_found = True
                print(f"Текст '{target_block}' найден. Ищем следующую таблицу...")
                # continue 
        
        # Шаг 2: После того как текст найден, ищем ПЕРВУЮ встречную таблицу
        if target_found and element.tag.endswith('tbl'):
            # Считаем, сколько таблиц встретилось в XML до текущего момента
            # Это и будет порядковый индекс в doc.tables
            table_index = len([e for e in doc.element.body[:i] if e.tag.endswith('tbl')])
            
            print(f"Таблица найдена (индекс в документе: {table_index})")
            return doc.tables[table_index]

    if not target_found:
        raise ValueError(f"Текст '{target_block}' не найден в документе")
    else:
        raise ValueError(f"Текст найден, но после него нет ни одной таблицы")

def table_to_dataframe(table):
    data = []
    for row in table.rows:
        current_row = []
        for cell in row.cells:
            text = cell.text.strip()
            current_row.append(text)
        data.append(current_row)
    
    # Создаем DataFrame
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

In [6]:
try:
    df = {}
    target_block = "Заболеваемость и контингенты пациентов активным туберкулезом\nпо субъектам Российской Федерации"
    for year in years:
        table = extract_tuberculosis_table(original_url[year], target_block)
        df[year] = table_to_dataframe(table)
        display(year)
        display(df[year])
        #df.to_csv('tuberculosis_data.csv', index=False, encoding='utf-8-sig')
except Exception as e:
    print(f"Произошла ошибка: {str(e)}")

Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 4)


2016

,СУБЪЕКТЫ ФЕДЕРАЦИИ,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов с впервые в жизни установ.диагнозом,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года
0,СУБЪЕКТЫ ФЕДЕРАЦИИ,абсолютные числа,абсолютные числа,на 100000 соот.населения,на 100000 соот.населения,абсолютные числа,абсолютные числа,на 100000 соот.населения,на 100000 соот.населения
1,СУБЪЕКТЫ ФЕДЕРАЦИИ,2015,2016,2015,2016,2015,2016,2015,2016
2,Российская Федерация,84515,78121,"57,7","53,3",189186,178080,"129,1","121,5"
3,Центральный федеральный округ,14700,13375,"37,7","34,2",26988,23615,"69,0","60,4"
4,Белгородская область,420,333,"27,1","21,5",564,428,"36,4","27,6"
...,...,...,...,...,...,...,...,...,...
92,Магаданская область,109,82,"74,0","56,0",228,174,"155,8","118,9"
93,Сахалинская область,320,306,"65,6","62,8",1177,1109,"241,5","227,6"
94,Еврейская автономная область,209,202,"125,0","121,6",592,500,"356,4","301,0"
95,Чукотский автономный округ,79,86,"156,9","171,5",168,187,"334,9","372,8"


Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 5)


2017

...

2019

,СУБЪЕКТЫ ФЕДЕРАЦИИ,число пациентов с впервые в жизни установленным диагнозом,число пациентов с впервые в жизни установленным диагнозом,число пациентов с впервые в жизни установленным диагнозом,число пациентов с впервые в жизни установленным диагнозом,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года
0,СУБЪЕКТЫ ФЕДЕРАЦИИ,абс. число,абс. число,на 100000 соот.населения,на 100000 соот.населения,абс. число,абс. число,на 100000 соот.населения,на 100000 соот.населения
1,СУБЪЕКТЫ ФЕДЕРАЦИИ,2018,2019,2018,2019,2018,2019,2018,2019
2,Российская Федерация,65234,60531,"44,4","41,2",149182,126737,"101,6","86,3"
3,Центральный федеральный округ,10952,10023,"27,8","25,5",18019,12970,"45,8","32,9"
4,Белгородская область,265,220,"17,1","14,2",307,257,"19,8","16,6"
...,...,...,...,...,...,...,...,...,...
92,Магаданская область,57,54,"40,0","38,2",153,139,"108,3","98,4"
93,Сахалинская область,259,243,"52,9","49,6",972,825,"198,5","168,5"
94,Еврейская автономная область,169,147,"105,0","91,9",436,411,"272,6","257,0"
95,Чукотский автономный округ,93,68,"187,9","136,9",260,233,"523,5","469,2"


Текст 'Заболеваемость и контингенты пациентов активным туберкулезом
по субъектам Российской Федерации' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 5)


2020

,СУБЪЕКТЫ ФЕДЕРАЦИИ,число пациентов с впервые в жизни установленным диагнозом,число пациентов с впервые в жизни установленным диагнозом,число пациентов с впервые в жизни установленным диагнозом,число пациентов с впервые в жизни установленным диагнозом,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года,число пациентов состоящих под диспансерным наблюдением на конец года
0,СУБЪЕКТЫ ФЕДЕРАЦИИ,абс. число,абс. число,на 100000 соот.населения,на 100000 соот.населения,абс. число,абс. число,на 100000 соот.населения,на 100000 соот.населения
1,СУБЪЕКТЫ ФЕДЕРАЦИИ,2019,2020,2019,2020,2019,2020,2019,2020
2,Российская Федерация,60531,47399,"41,2","32,3",126737,102785,"86,4","70,0"
3,Центральный федеральный округ,10023,7667,"25,4","19,4",12970,9508,"32,9","24,1"
4,Белгородская область,220,183,"14,2","11,8",257,235,"16,6","15,2"
...,...,...,...,...,...,...,...,...,...
92,Магаданская область,54,40,"38,4","28,5",139,95,"99,2","67,8"
93,Сахалинская область,243,253,"49,7","51,8",825,645,169,"132,1"
94,Еврейская автономная область,147,97,"92,4","61,3",411,364,"259,6","229,9"
95,Чукотский автономный округ,68,63,"136,1","125,3",233,166,"463,3","330,1"
